# Black-box frequency inference (cache): model differences

Compares Macro-F1 and OW-Acc (ordinal-weighted accuracy) between the three
models (smol, qwen, granite) for PG19 and PubMed. For each dataset it reports
the difference for every pair of models and the largest absolute difference
per metric.

Scores are read from `<dataset>_<model>_cache_black_box_freq_results/metrics.json`.

In [ ]:
import json
from itertools import combinations
from pathlib import Path

import pandas as pd

RESULTS_ROOT = Path(".")
OUTPUT_DIRECTORY = Path("black_box_model_differences_cache_results")

DATASETS = ("pg19", "pubmed")
DATASET_LABELS = {"pg19": "PG19", "pubmed": "PubMed"}
MODELS = ("smol", "qwen", "granite")

METRICS = {
    "Macro-F1": ("normal_metrics", "macro_f1"),
    "OW-Acc": ("ordinal_weighted_metrics", "ow_accuracy"),
}

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 1. Scores per dataset and model

In [ ]:
def load_scores() -> pd.DataFrame:
    rows = []
    for dataset in DATASETS:
        for model in MODELS:
            metrics_path = (
                RESULTS_ROOT
                / f"{dataset}_{model}_cache_black_box_freq_results"
                / "metrics.json"
            )
            with metrics_path.open(encoding="utf-8") as handle:
                metrics = json.load(handle)
            row = {"dataset": DATASET_LABELS[dataset], "model": model}
            for metric_name, (section, key) in METRICS.items():
                row[metric_name] = metrics[section][key]
            rows.append(row)
    return pd.DataFrame(rows)


scores = load_scores()
scores.to_csv(OUTPUT_DIRECTORY / "scores.csv", index=False)
scores.round(4)

## 2. Pairwise differences

`difference = model_a - model_b`. A positive value means `model_a` scores
higher. `abs_difference` is used to rank the pairs.

In [ ]:
def pairwise_differences(scores: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset, dataset_scores in scores.groupby("dataset", sort=False):
        by_model = dataset_scores.set_index("model")
        for model_a, model_b in combinations(MODELS, 2):
            for metric_name in METRICS:
                score_a = by_model.loc[model_a, metric_name]
                score_b = by_model.loc[model_b, metric_name]
                rows.append(
                    {
                        "dataset": dataset,
                        "metric": metric_name,
                        "model_a": model_a,
                        "model_b": model_b,
                        "score_a": score_a,
                        "score_b": score_b,
                        "difference": score_a - score_b,
                        "abs_difference": abs(score_a - score_b),
                    }
                )
    return pd.DataFrame(rows)


differences = pairwise_differences(scores)
differences.to_csv(OUTPUT_DIRECTORY / "pairwise_differences.csv", index=False)
differences.round(4)

## 3. Highest difference per dataset and metric

In [ ]:
highest = differences.loc[
    differences.groupby(["dataset", "metric"], sort=False)["abs_difference"].idxmax()
].reset_index(drop=True)
highest.to_csv(OUTPUT_DIRECTORY / "highest_differences.csv", index=False)

for row in highest.itertuples():
    better, worse = (
        (row.model_a, row.model_b) if row.difference >= 0 else (row.model_b, row.model_a)
    )
    print(
        f"{row.dataset:<7} {row.metric:<9} largest gap: {better} vs {worse} "
        f"= {row.abs_difference:.4f} ({row.abs_difference * 100:.2f} pp)"
    )

print(f"\nResults written to: {OUTPUT_DIRECTORY.resolve()}")
highest.round(4)